# S5.G1-1 · G3 五臂对比 Santander：数据、调参与本地臂

按冻结的预注册（`modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml`，S5.G0 提交 f80f74f）执行 G3 第一批的 Santander 部分。
本 notebook 只做数据准备、调参与本地臂（L0、L1 k=5/10/20、B_T1、B_T2、集中式代理 L4、阴性对照 NC）的训练与打分；
联邦臂 A 在 S5.G1-2 用隐语 SGB 训练，配对比较在 S5.G1-3。

- 原始数据为公开竞赛数据 Santander Product Recommendation，只在维护者本机；本 notebook 读取的是按冻结规则筛出的私有行文件（sha256 核对），
  行数据、客户号、学到的编码与分段边界、逐行打分只写入私有目录，**不入仓库**。
- 首格核对预注册状态为 frozen、组件指纹与冻结记录一致，不一致即中止。
- 本 notebook 不输出开发评估集上的任何指标；指标只在 S5.G1-3 计算一次。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g1_santander_run.yaml"
run = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]


def sha(path):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:HASH_PREFIX_LEN]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("seed:", run["seed"], "| seeds:", run["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", run["step_id"], "|", run["purpose"])
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = run["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g1_santander_run.yaml sha256:09834fbb00ea
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: f80f74f（工作区含本步未提交改动）
step: S5.G1 | 按冻结的 S6.P3 在 Santander 上训练 L0、L1（k=5/10/20）、B_T1、B_T2、A（SGB）、集中式代理 L4 与阴性对照 NC，并做配对比较
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_santander as G

prep = fz["santander_prep"]
src = PRIV / run["selected_rows_file"]
assert hashlib.sha256(src.read_bytes()).hexdigest() == prep["selected_rows_sha256"]
rows = pd.read_csv(src, dtype=str, keep_default_na=False)


def feature_months(name):
    return G.months_between(*prep["groups"][name]["feature_months"])


tr = G.build_split(rows[rows[G.GROUP_COL] == "train"], feature_months("train"))
iv = G.build_split(rows[rows[G.GROUP_COL] == "inner_val"], feature_months("inner_val"))
cand = rows[rows[G.GROUP_COL] == run["candidate_group"]]
cand_ids = pd.to_numeric(cand[G.ID_COL].str.strip()).astype(np.int64)
bucket = {c: G.customer_bucket(c, prep["salt"], prep["n_buckets"]) for c in cand_ids.unique()}
lo, hi = prep["groups"]["dev_eval"]["buckets"]
dev_rows = cand[cand_ids.map(bucket).between(lo, hi - 1).to_numpy()]
de = G.build_split(dev_rows, feature_months("dev_eval"))
counts = pd.DataFrame(G.summary_counts([("train", tr), ("inner_val", iv), ("dev_eval", de)]))
print(counts.to_string(index=False))
n_pos_dev = int(de.loc[de[G.LABEL] == 1, G.ID_COL].nunique())
target = pre["datasets"]["santander"]["eval_positives_target"]
print("开发评估集按客户去重的正例：%d（预注册目标 %d）；每位客户一行：%s" % (n_pos_dev, target, de[G.ID_COL].is_unique))

    group  rows  customers  positives  positive_customers
    train 98956       9301       3740                1851
inner_val 36863       9338       1215                 971
 dev_eval 20351      20351        586                 586
开发评估集按客户去重的正例：586（预注册目标 700）；每位客户一行：True


**读法**：

- 训练组 9,301 位客户、98,956 个「本月 → 次月」配对，正例率 3.78%；内层验证组 9,338 位客户；开发评估集 20,351 位客户、每位一行。
- **开发评估集按客户去重只有 586 个正例，低于预注册目标 700。** 样本量规则用的是训练月份的平均正例率 3.78%，而评估月（2016-03 → 2016-04）的正例率只有 2.9%。
- 按预注册停止规则 5：不追加样本重评；主对比按实际可检出差异报告，正例不足的影响在 S5.G1-3 的「可检出差异」一栏体现。
- 这同时触发 DR-GOV-016 · D-2 的推翻条件：后续步骤先做 HSBC mock 三档（S5.G3），再做 SS-XGB（S5.G2）。

In [3]:
from modules.m5_modeling.components import g3_arms as A

enc = G.fit_encoder(tr)
X = {}
for name, df in (("train", tr), ("inner_val", iv), ("dev_eval", de)):
    xg, xh = G.encode(df, enc)
    X[name] = {"xg": xg, "xh": xh, "y": df[G.LABEL].to_numpy(), "yhk": df[G.HK_LABEL].to_numpy(),
               "cust": df[G.ID_COL].to_numpy()}
np.savez_compressed(PRIV / run["arrays_file"], **{n + "__" + k: v for n, d in X.items() for k, v in d.items()})
print({n: (d["xg"].shape, d["xh"].shape) for n, d in X.items()})

# —— 各臂的内地侧特征集 ——
cols = {name: G.GUEST_FEATURES.index(col) for name, col in run["segment_columns"].items()}
cols.update({c: G.GUEST_FEATURES.index(c) for c in run["segment_categorical"]})
edges = A.segment_edges(X["train"]["xg"], cols, ad["l1_segments"]["quantile_bins"])
cat_sizes = {c: len(enc[c]) for c in run["segment_categorical"]}
seg = {n: A.segment_ids(X[n]["xg"], cols, edges, cat_sizes) for n in X}
SETS, RELEASE = {}, {}
for k in run["l1_k_values"]:
    table, gmean, info = A.k_anonymous_means(seg["train"], X["train"]["cust"], X["train"]["xh"], k)
    SETS["L1_k%d" % k] = {n: np.hstack([X[n]["xg"], A.apply_segment_means(seg[n], table, gmean)]) for n in X}
    RELEASE["L1_k%d" % k] = info


def bt1_sets(seed):
    b = ad["b_t1"]
    emb = A.tree_embedding(X["train"]["xh"], b["n_trees"], b["depth"], seed)
    codes = {n: A.leaf_codes(emb, X[n]["xh"]) for n in X}
    allowed = A.k_coarsen(codes["train"], X["train"]["cust"], b["k"])
    info = {"leaves_released": int(sum(len(a) for a in allowed))}
    return {n: np.hstack([X[n]["xg"], A.one_hot_codes(codes[n], allowed)]) for n in X}, info


SETS["B_T1"], RELEASE["B_T1"] = bt1_sets(SEED0)
hk_model = A.xgb_fit(X["train"]["xh"], X["train"]["yhk"], ad["b_t2"]["config"], MB, SEED0, NJ)
SETS["B_T2"] = {n: np.column_stack([X[n]["xg"], A.xgb_score(hk_model, X[n]["xh"])]) for n in X}
SETS["L0"] = {n: X[n]["xg"] for n in X}
SETS["L4"] = {n: np.hstack([X[n]["xg"], X[n]["xh"]]) for n in X}
print("各臂内地侧特征数：", {a: s["train"].shape[1] for a, s in SETS.items()})
print("k-匿名发布：", RELEASE)

{'train': ((98956, 19), (98956, 25)), 'inner_val': ((36863, 19), (36863, 25)), 'dev_eval': ((20351, 19), (20351, 25))}


各臂内地侧特征数： {'L1_k5': 44, 'L1_k10': 44, 'L1_k20': 44, 'B_T1': 78, 'B_T2': 20, 'L0': 19, 'L4': 44}
k-匿名发布： {'L1_k5': {'segments_total': 87, 'segments_released': 77, 'segments_suppressed': 10}, 'L1_k10': {'segments_total': 87, 'segments_released': 70, 'segments_suppressed': 17}, 'L1_k20': {'segments_total': 87, 'segments_released': 60, 'segments_suppressed': 27}, 'B_T1': {'leaves_released': 59}}


**读法**：

- 内地 19 列、香港 25 列，与 10/21 汇报和 HSBC mock 的业务切分一致。
- L1 的分段（年龄 5 分位 × 收入 4 分位 × 客群类别）共 87 段；k = 5 / 10 / 20 时分别有 10 / 17 / 27 段因不同客户数不足 k 被抑制，回落到全体均值。
- B_T1 的 10 棵随机树共有 59 个叶子在 k = 10 下可以发布，独热后给内地 59 列。
- 这些计数是本样本的发布统计，不含任何分段内的数值。

In [4]:
configs = A.sample_configs(ad["search_space"], pre["tuning"]["configs_per_arm"], pre["seeds"]["tuning"])
TUNING = {}
t0 = time.perf_counter()
for arm in ["L0", "L1_k5", "L1_k10", "L1_k20", "B_T1", "B_T2", "L4"]:
    best, table = A.tune(SETS[arm]["train"], X["train"]["y"], SETS[arm]["inner_val"], X["inner_val"]["y"],
                         configs, MB, SEED0, NJ)
    TUNING[arm] = {"best": best, "best_val_auc": max(r["val_auc"] for r in table), "candidates": table}
TUNING["A"] = {"best": TUNING["L4"]["best"], "source": "L4 集中式明文代理（freeze A-4）"}
b_pool = ["B_T1", "B_T2", "L1_k10"]
TUNING["B_best"] = max(b_pool, key=lambda a: TUNING[a]["best_val_auc"])
(PRIV / run["tuning_file"]).write_text(json.dumps(TUNING, ensure_ascii=False), encoding="utf-8")
view = pd.DataFrame([dict(TUNING[a]["best"], arm=a, inner_val_auc=TUNING[a]["best_val_auc"])
                     for a in ["L0", "L1_k5", "L1_k10", "L1_k20", "B_T1", "B_T2", "L4"]])
print(view.round(run["round_auc"]).to_string(index=False))
print("B_best（开发评估前按内层验证集选定）：", TUNING["B_best"], "| 调参耗时 %.0f 秒" % (time.perf_counter() - t0))

 learning_rate  max_depth  num_boost_round  reg_lambda    arm  inner_val_auc
          0.10          5               50         0.1     L0         0.8180
          0.10          5               50         0.1  L1_k5         0.8177
          0.20          3               50         1.0 L1_k10         0.8175
          0.10          5               50         0.1 L1_k20         0.8160
          0.10          5              150        10.0   B_T1         0.8828
          0.05          5              150         1.0   B_T2         0.8482
          0.10          5              150        10.0     L4         0.8836
B_best（开发评估前按内层验证集选定）： B_T1 | 调参耗时 95 秒


**读法**（内层验证集，只用于选配置）：

- L1 三档与 L0 几乎相同（约 0.818）：香港按分段给出的均值没有带来可见的信息。
- **B_T1（香港随机树嵌入，一次性发给内地）达到 0.883，与集中式代理 L4 的 0.884 几乎相同**；B_T2（香港自有标签的分数）为 0.848。
- 按预注册，B_best 在开发评估前按内层验证集选定为 B_T1。
- 这些是选配置用的数，结论以 S5.G1-3 在开发评估集上的配对比较为准。

In [5]:
SCORES, COST = {}, {}
for arm in ["L0", "L1_k5", "L1_k10", "L1_k20", "B_T2", "L4"]:
    t0 = time.perf_counter()
    m = A.xgb_fit(SETS[arm]["train"], X["train"]["y"], TUNING[arm]["best"], MB, SEED0, NJ)
    COST[arm] = time.perf_counter() - t0
    SCORES[arm] = A.xgb_score(m, SETS[arm]["dev_eval"])
bt1 = []
for s in run["seeds"]:
    sets_s, _ = bt1_sets(s)
    m = A.xgb_fit(sets_s["train"], X["train"]["y"], TUNING["B_T1"]["best"], MB, s, NJ)
    bt1.append(A.xgb_score(m, sets_s["dev_eval"]))
SCORES["B_T1"] = np.vstack(bt1)
nc = []
for s in run["seeds"]:
    sh = {n: np.hstack([X[n]["xg"], A.shuffle_rows(X[n]["xh"], s)]) for n in ("train", "dev_eval")}
    m = A.xgb_fit(sh["train"], X["train"]["y"], TUNING["A"]["best"], MB, SEED0, NJ)
    nc.append(A.xgb_score(m, sh["dev_eval"]))
SCORES["NC"] = np.vstack(nc)
np.savez_compressed(PRIV / run["local_scores_file"], **SCORES)
print("已保存各臂开发评估集打分（私有）：", {a: np.shape(v) for a, v in SCORES.items()})
print("单次拟合耗时（秒）：", {a: round(v) for a, v in COST.items()})

已保存各臂开发评估集打分（私有）： {'L0': (20351,), 'L1_k5': (20351,), 'L1_k10': (20351,), 'L1_k20': (20351,), 'B_T2': (20351,), 'L4': (20351,), 'B_T1': (6, 20351), 'NC': (6, 20351)}
单次拟合耗时（秒）： {'L0': 0, 'L1_k5': 0, 'L1_k10': 0, 'L1_k20': 0, 'B_T2': 1, 'L4': 1}


**读法**：确定性臂（L0、L1、B_T2、L4）各拟合一次；B_T1 与 NC 各跑 6 个 seed（嵌入与打乱的随机性）。打分只写入私有目录。